# Task 7: Long Sum

Bot task id: `07_long_sum`, deadline 06.12.2026 00:00 MSK. [Open in Colab](https://colab.research.google.com/github/PE51K/itmo-ai-talent-hub-efficient-models-course/blob/main/hw3/07_long_sum.ipynb)

## Условие

Сумма по строкам матрицы `x` формы `(N0, T)`:

$$out[i] = \sum_{j=0}^{T-1} x[i, j], \quad i = 0 \dots N_0-1$$

**Требования**
- сетка одномерная: `pid0` перебирает строки `i`;
- внутри ядра цикл по тайлам длины `B1 < T`;
- на каждом шаге загружайте блок `tl.arange(0, B1)` и аккумулируйте сумму;
- маска для хвостового блока, если `T` не делится на `B1`;
- решение должно работать на GPU (CUDA).

**Подсказка:** `row = pid0`, `offs = start + tl.arange(0, B1)`, глобальные смещения `row * T + offs`; копите сумму в `acc` и после цикла сохраните в `out[row]`.

Проверка: `x` формы `(4, 200)`.

## Setup

Runtime -> Change runtime type -> T4 GPU.

In [ ]:
import importlib.util

import torch
import triton

assert torch.cuda.is_available(), "no GPU in this runtime: Runtime -> Change runtime type -> T4 GPU"
print(f"torch {torch.__version__}, triton {triton.__version__}, {torch.cuda.get_device_name()}")

## Solution

`%%writefile` saves the cell to `07_long_sum.py`, the file to submit. Run it after every edit, then the check.

In [ ]:
%%writefile 07_long_sum.py
import torch
import triton
import triton.language as tl


@triton.jit
def _long_sum_kernel(x_ptr, out_ptr, N0, T, B1: tl.constexpr):
    # YOUR CODE HERE
    pass


def sum_triton(x: torch.Tensor) -> torch.Tensor:
    # YOUR CODE HERE
    raise NotImplementedError

## Check

Compares with PyTorch on shapes with tails and checks some of the requirements. These are my tests, not the bot's: passing them doesn't guarantee the bot accepts the solution.

In [ ]:
PATH = "07_long_sum.py"
spec = importlib.util.spec_from_file_location("sol", PATH)
sol = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sol)
SRC = open(PATH).read()

x = torch.randn(4, 200, device="cuda")
torch.testing.assert_close(sol.sum_triton(x), x.sum(1), atol=1e-4, rtol=1e-4)

torch.cuda.synchronize()
print("PASS")

## Download

In [ ]:
from google.colab import files

files.download("07_long_sum.py")